import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Load dataset
df = pd.read_csv(r"C:\Users\mages\capstone\smartcrop-system\data\raw\crop_recommendation.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

In [4]:
# Calculate thresholds
rainfall_threshold = df['rainfall'].quantile(0.25)
temp_threshold = df['temperature'].quantile(0.75)

print(f"Rainfall threshold (low): {rainfall_threshold:.2f} mm")
print(f"Temperature threshold (high): {temp_threshold:.2f} °C")

# Create failure risk column
# Logic: Low rainfall AND high temperature = crop failure risk
df['crop_failure_risk'] = (
    (df['rainfall'] < rainfall_threshold) & 
    (df['temperature'] > temp_threshold)
).astype(int)

print(f"\nHigh Risk samples: {df['crop_failure_risk'].sum()}")
print(f"Safe samples: {(df['crop_failure_risk']==0).sum()}")
print(f"\nRisk distribution:\n{df['crop_failure_risk'].value_counts()}")

Rainfall threshold (low): 64.55 mm
Temperature threshold (high): 28.56 °C

High Risk samples: 181
Safe samples: 2019

Risk distribution:
crop_failure_risk
0    2019
1     181
Name: count, dtype: int64


In [5]:
# Features we'll use for prediction
features = ['N', 'P', 'K', 'temperature', 
            'humidity', 'ph', 'rainfall']

# Normalize to 0-1 scale
scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

print("Before scaling - Rainfall range:")
print(f"Min: {df['rainfall'].min()}, Max: {df['rainfall'].max()}")

print("\nAfter scaling - Rainfall range:")
print(f"Min: {df_scaled['rainfall'].min():.2f}, Max: {df_scaled['rainfall'].max():.2f}")

print("\nFirst 5 rows after scaling:")
print(df_scaled[features].head())

Before scaling - Rainfall range:
Min: 20.21126747, Max: 298.5601175

After scaling - Rainfall range:
Min: 0.00, Max: 1.00

First 5 rows after scaling:
          N         P      K  temperature  humidity        ph  rainfall
0  0.642857  0.264286  0.190     0.345886  0.790267  0.466264  0.656458
1  0.607143  0.378571  0.180     0.371445  0.770633  0.549480  0.741675
2  0.428571  0.357143  0.195     0.406854  0.793977  0.674219  0.875710
3  0.528571  0.214286  0.175     0.506901  0.768751  0.540508  0.799905
4  0.557143  0.264286  0.185     0.324378  0.785626  0.641291  0.871231


In [6]:
# Convert crop names to numbers
le = LabelEncoder()
df_scaled['crop_encoded'] = le.fit_transform(df['label'])

print("Crop label encoding:")
for i, crop in enumerate(le.classes_):
    print(f"{i} → {crop}")

Crop label encoding:
0 → apple
1 → banana
2 → blackgram
3 → chickpea
4 → coconut
5 → coffee
6 → cotton
7 → grapes
8 → jute
9 → kidneybeans
10 → lentil
11 → maize
12 → mango
13 → mothbeans
14 → mungbean
15 → muskmelon
16 → orange
17 → papaya
18 → pigeonpeas
19 → pomegranate
20 → rice
21 → watermelon


In [7]:
import os
os.makedirs(r"C:\Users\mages\capstone\smartcrop-system\data\processed", exist_ok=True)

# Save final processed dataset
df_scaled.to_csv(r"C:\Users\mages\capstone\smartcrop-system\data\processed\smartcrop_features.csv", 
                 index=False)

# Verify it saved
saved = pd.read_csv(r"C:\Users\mages\capstone\smartcrop-system\data\processed\smartcrop_features.csv")
print("✅ Saved successfully!")
print(f"Shape: {saved.shape}")
print(f"\nColumns: {saved.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(saved.head(3))

✅ Saved successfully!
Shape: (2200, 10)

Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label', 'crop_failure_risk', 'crop_encoded']

First 3 rows:
          N         P      K  temperature  humidity        ph  rainfall label  \
0  0.642857  0.264286  0.190     0.345886  0.790267  0.466264  0.656458  rice   
1  0.607143  0.378571  0.180     0.371445  0.770633  0.549480  0.741675  rice   
2  0.428571  0.357143  0.195     0.406854  0.793977  0.674219  0.875710  rice   

   crop_failure_risk  crop_encoded  
0                  0            20  
1                  0            20  
2                  0            20  


## Preprocessing Summary

### Steps Completed:
1. Loaded raw crop recommendation dataset (2200 samples, 8 columns)
2. Checked for missing values — none found
3. Created crop_failure_risk target variable using:
   - Low rainfall (below 25th percentile) AND
   - High temperature (above 75th percentile)
4. Normalized all numerical features to 0-1 scale using MinMaxScaler
5. Encoded crop labels to numerical values using LabelEncoder
6. Saved processed dataset to data/processed/smartcrop_features.csv

### Final Dataset:
- Total samples: 2200
- Features: N, P, K, temperature, humidity, ph, rainfall
- Target: crop_failure_risk (0=Safe, 1=High Risk)
- Ready for ML model training ✅